# Cell 1 — Setup

In [5]:
%load_ext autoreload
%autoreload 2

import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef, confusion_matrix,
)

RANDOM_STATE = 25
DATA_DIR    = Path.cwd().parent / 'data' / 'processed'
RESULTS_DIR = Path.cwd().parent / 'outputs' / 'results'
FIG_DIR     = Path.cwd().parent / 'outputs' / 'figures'
MODELS_DIR  = Path.cwd().parent / 'models'
for d in [RESULTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Cell 2 — Load winning Phase 4 model + splits

In [6]:
# Phase 4 winner: XGBoost + RandomOver at CV macro-F1 = 0.5188
best_pipe = joblib.load(MODELS_DIR / 'phase4_best_XGBoost_RandomOver.joblib')

X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
X_test  = pd.read_parquet(DATA_DIR / 'X_test.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']
y_test  = pd.read_parquet(DATA_DIR / 'y_test.parquet') ['Victims_Condition']

# XGBoost was trained on integer labels — match the encoding
y_train_int, class_labels = pd.factorize(y_train, sort=True)
y_test_int = pd.Categorical(y_test, categories=class_labels).codes

print(f"Model: {best_pipe.named_steps['clf'].__class__.__name__}")
print(f"X_test: {X_test.shape}")
print(f"Class order (integer → label): {dict(enumerate(class_labels))}")

Model: XGBClassifier
X_test: (92617, 21)
Class order (integer → label): {0: 'With dead victims', 1: 'With injured victims', 2: 'Without victims'}


# Cell 3 — Final test-set evaluation (one-and-only touch)

In [7]:
print("Predicting on sealed test set...")
t0 = time.time()
y_pred_int = best_pipe.predict(X_test)
print(f"  prediction time: {time.time() - t0:.1f}s")

acc = accuracy_score(y_test_int, y_pred_int)
macro_f1 = f1_score(y_test_int, y_pred_int, average='macro', zero_division=0)
mcc = matthews_corrcoef(y_test_int, y_pred_int)
per_class_f1 = f1_score(y_test_int, y_pred_int, average=None,
                        labels=[0, 1, 2], zero_division=0)

print(f"\n=== FINAL TEST-SET METRICS (Phase 4 winner: XGBoost + RandomOver) ===")
print(f"Accuracy : {acc:.4f}")
print(f"Macro-F1 : {macro_f1:.4f}")
print(f"MCC      : {mcc:.4f}")
for cls_name, f1 in zip(class_labels, per_class_f1):
    print(f"  F1 [{cls_name}]: {f1:.4f}")

pd.DataFrame([{
    'model':      'XGBoost + RandomOver (Phase 4 winner)',
    'accuracy':   acc,
    'macro_f1':   macro_f1,
    'mcc':        mcc,
    'f1_dead':    per_class_f1[0],
    'f1_injured': per_class_f1[1],
    'f1_without': per_class_f1[2],
}]).to_csv(RESULTS_DIR / 'phase5_final_test_metrics.csv', index=False)

cm = confusion_matrix(y_test_int, y_pred_int, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm,
                     index=[f'true_{c}' for c in class_labels],
                     columns=[f'pred_{c}' for c in class_labels])
print("\nConfusion matrix:")
print(cm_df)
cm_df.to_csv(RESULTS_DIR / 'phase5_final_confusion_matrix.csv')

Predicting on sealed test set...
  prediction time: 1.7s

=== FINAL TEST-SET METRICS (Phase 4 winner: XGBoost + RandomOver) ===
Accuracy : 0.6306
Macro-F1 : 0.5194
MCC      : 0.2784
  F1 [With dead victims]: 0.3607
  F1 [With injured victims]: 0.7336
  F1 [Without victims]: 0.4638

Confusion matrix:
                           pred_With dead victims  pred_With injured victims  \
true_With dead victims                       3141                       2229   
true_With injured victims                    6973                      44450   
true_Without victims                         1034                       8063   

                           pred_Without victims  
true_With dead victims                      899  
true_With injured victims                 15013  
true_Without victims                      10815  


# Cell 4 — Extract preprocessed test data + stratified 10k sample

In [8]:
preprocessor = best_pipe.named_steps['pre']
classifier   = best_pipe.named_steps['clf']

X_test_transformed = preprocessor.transform(X_test)
feature_names_ohe  = preprocessor.get_feature_names_out()

# Sklearn's built-in stratified sampling — one call
_, sample_idx = train_test_split(
    np.arange(len(y_test_int)),
    test_size    = 10_000,
    stratify     = y_test_int,
    random_state = RANDOM_STATE,
)
X_sample = X_test_transformed[sample_idx]
y_sample = y_test_int[sample_idx]

print(f"Post-preprocessing shape: {X_test_transformed.shape}")
print(f"Number of OHE features: {len(feature_names_ohe)}")
print(f"Stratified sample: {len(sample_idx)} rows")
print(f"Sample class distribution: {pd.Series(y_sample).value_counts().to_dict()}")

Post-preprocessing shape: (92617, 398)
Number of OHE features: 398
Stratified sample: 10000 rows
Sample class distribution: {1: 7173, 2: 2150, 0: 677}


# Cell 5 — TreeSHAP on stratified 10k sample (fast pass)

In [9]:
print("Initialising TreeSHAP explainer...")
explainer = shap.TreeExplainer(classifier)

print("Computing SHAP values on 10k sample...")
t0 = time.time()
# Multiclass returns shape (n_samples, n_features, n_classes)
shap_values_sample = explainer.shap_values(X_sample)
print(f"  done in {(time.time() - t0)/60:.1f} min")
print(f"  SHAP values shape: {shap_values_sample.shape}")

np.savez_compressed(
    RESULTS_DIR / 'phase5_shap_values_sample.npz',
    shap_values   = shap_values_sample,
    sample_idx    = sample_idx,
    feature_names = feature_names_ohe,
)
print("Saved: outputs/results/phase5_shap_values_sample.npz")

Initialising TreeSHAP explainer...
Computing SHAP values on 10k sample...
  done in 2.6 min
  SHAP values shape: (10000, 398, 3)
Saved: outputs/results/phase5_shap_values_sample.npz


# Cell 6 — Aggregate raw OHE SHAP values back to the 21 original features

In [10]:
def map_ohe_to_original(feature_name):
    """Strip 'num__' or 'cat__' prefix, then remove the OHE category suffix."""
    if feature_name.startswith('num__'):
        return feature_name[5:]
    if feature_name.startswith('cat__'):
        rest = feature_name[5:]
        parts = rest.rsplit('_', 1)
        return parts[0] if len(parts) == 2 else rest
    return feature_name

original_names   = [map_ohe_to_original(f) for f in feature_names_ohe]
unique_originals = list(dict.fromkeys(original_names))

def aggregate_shap(shap_arr):
    """(n_samples, n_ohe, n_classes) → (n_samples, n_original, n_classes)"""
    agg = np.zeros((shap_arr.shape[0], len(unique_originals), shap_arr.shape[2]))
    for orig_idx, orig_name in enumerate(unique_originals):
        cols = [i for i, n in enumerate(original_names) if n == orig_name]
        agg[:, orig_idx, :] = shap_arr[:, cols, :].sum(axis=1)
    return agg

shap_agg_sample = aggregate_shap(shap_values_sample)
print(f"Aggregated shape: {shap_agg_sample.shape}")
print(f"Original features ({len(unique_originals)}): {unique_originals}")

Aggregated shape: (10000, 22, 3)
Original features (22): ['Road_Km', 'Latitude', 'Longitude', 'Hour', 'Month', 'State', 'Road_ID', 'Road_ID_NO', 'Accident_Type', 'Day_Phase', 'Km_Direction', 'Weather_Condition', 'Road_Type', 'Road_Delineation', 'Police_Station', 'Missing_Police_Station', 'Missing_Road_Data', 'Cause_Group', 'Weekday', 'Day_Type', 'Region_BR', 'Time_Period']


# Cell 7 — Global SHAP plots (aggregated features)

In [11]:
mean_abs_shap = np.abs(shap_agg_sample).mean(axis=(0, 2))
importance_df = (pd.DataFrame({
        'feature': unique_originals,
        'mean_abs_shap': mean_abs_shap,
    })
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True))

importance_df.to_csv(RESULTS_DIR / 'phase5_feature_importance.csv', index=False)
print("Top 10 features by mean |SHAP|:")
print(importance_df.head(10).to_string(index=False))

# Plot 1: Global bar
top_n = 15
top = importance_df.head(top_n).iloc[::-1]  # reverse for horizontal bar
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top['feature'], top['mean_abs_shap'], color='#2E74B5')
ax.set_xlabel('mean |SHAP value|')
ax.set_title(f'Global feature importance (top {top_n}) — XGBoost + RandomOver')
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase5_global_bar.png', dpi=120, bbox_inches='tight')
plt.close()
print("\nSaved: outputs/figures/phase5_global_bar.png")

# Plot 2: Global beeswarm
# Aggregate feature values from OHE (mean across each feature's OHE columns)
X_sample_dense = X_sample.toarray() if hasattr(X_sample, 'toarray') else X_sample
X_agg_values = np.zeros((X_sample_dense.shape[0], len(unique_originals)))
for orig_idx, orig_name in enumerate(unique_originals):
    cols = [i for i, n in enumerate(original_names) if n == orig_name]
    X_agg_values[:, orig_idx] = X_sample_dense[:, cols].mean(axis=1)

shap.summary_plot(
    shap_agg_sample.mean(axis=2), X_agg_values,
    feature_names=unique_originals, max_display=15, show=False,
)
plt.title('SHAP beeswarm (avg across classes) — XGBoost + RandomOver')
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase5_global_beeswarm.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/phase5_global_beeswarm.png")

Top 10 features by mean |SHAP|:
         feature  mean_abs_shap
   Accident_Type       0.460920
        Latitude       0.169328
     Cause_Group       0.147606
  Police_Station       0.104932
       Longitude       0.104663
         Road_Km       0.098599
            Hour       0.083353
         Road_ID       0.072190
Road_Delineation       0.070487
       Day_Phase       0.068080

Saved: outputs/figures/phase5_global_bar.png
Saved: outputs/figures/phase5_global_beeswarm.png


# Cell 8 — Per-class SHAP importance (3 plots)

In [12]:
class_colors = ['#C00000', '#ED7D31', '#548235'] 

for cls_idx, cls_name in enumerate(class_labels):
    mean_abs = np.abs(shap_agg_sample[:, :, cls_idx]).mean(axis=0)
    df_cls = (pd.DataFrame({'feature': unique_originals, 'mean_abs_shap': mean_abs})
              .sort_values('mean_abs_shap', ascending=False)
              .head(15)
              .iloc[::-1])

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(df_cls['feature'], df_cls['mean_abs_shap'], color=class_colors[cls_idx])
    ax.set_xlabel('mean |SHAP value|')
    ax.set_title(f'Feature importance for class: {cls_name}')
    plt.tight_layout()

    safe_name = cls_name.replace(' ', '_')
    plt.savefig(FIG_DIR / f'phase5_per_class_{safe_name}.png',
                dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Saved: outputs/figures/phase5_per_class_{safe_name}.png")

Saved: outputs/figures/phase5_per_class_With_dead_victims.png
Saved: outputs/figures/phase5_per_class_With_injured_victims.png
Saved: outputs/figures/phase5_per_class_Without_victims.png


# Cell 9 — Dependence plots for top 3 features (fatal class)

In [13]:
fatal_importance = np.abs(shap_agg_sample[:, :, 0]).mean(axis=0)
top_3_indices    = np.argsort(fatal_importance)[::-1][:3]
top_3_features   = [unique_originals[i] for i in top_3_indices]
print(f"Top 3 features for fatal-class prediction: {top_3_features}")

for feat_idx, feat_name in zip(top_3_indices, top_3_features):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(X_agg_values[:, feat_idx],
               shap_agg_sample[:, feat_idx, 0],
               alpha=0.3, s=10, color='#C00000')
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.set_xlabel(f'{feat_name} (aggregated OHE value)')
    ax.set_ylabel('SHAP value for fatal class')
    ax.set_title(f'Dependence plot: {feat_name} → fatal-class contribution')
    plt.tight_layout()

    safe_name = feat_name.replace('/', '_').replace(' ', '_')
    plt.savefig(FIG_DIR / f'phase5_dependence_{safe_name}.png',
                dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Saved: outputs/figures/phase5_dependence_{safe_name}.png")

Top 3 features for fatal-class prediction: ['Accident_Type', 'Latitude', 'Cause_Group']
Saved: outputs/figures/phase5_dependence_Accident_Type.png
Saved: outputs/figures/phase5_dependence_Latitude.png
Saved: outputs/figures/phase5_dependence_Cause_Group.png


# Cell 10 — Waterfall plot: walk through one correctly-predicted fatal case

In [14]:
# Find a correctly-predicted fatal case
y_pred_sample = classifier.predict(X_sample)
correct_fatal_idx = np.where((y_pred_sample == 0) & (y_sample == 0))[0]
case_idx = correct_fatal_idx[0] if len(correct_fatal_idx) > 0 else np.where(y_sample == 0)[0][0]

# Build a SHAP Explanation object for this one case, fatal class
X_sample_row = X_sample[case_idx].toarray().flatten() \
               if hasattr(X_sample, 'toarray') else X_sample[case_idx]

explanation = shap.Explanation(
    values        = shap_values_sample[case_idx, :, 0],
    base_values   = explainer.expected_value[0],
    data          = X_sample_row,
    feature_names = feature_names_ohe,
)

shap.plots.waterfall(explanation, max_display=10, show=False)
plt.title('Waterfall plot: one correctly-predicted fatal case')
plt.tight_layout()
plt.savefig(FIG_DIR / 'phase5_waterfall_fatal.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved: outputs/figures/phase5_waterfall_fatal.png")

Saved: outputs/figures/phase5_waterfall_fatal.png


# Cell 11 — Full test set SHAP

In [15]:
print(f"Computing SHAP values on full test set ({X_test_transformed.shape[0]} rows)...")
t0 = time.time()
shap_values_full = explainer.shap_values(X_test_transformed)
print(f"  done in {(time.time() - t0)/60:.1f} min")
print(f"  Full SHAP values shape: {shap_values_full.shape}")

np.savez_compressed(
    RESULTS_DIR / 'phase5_shap_values_full.npz',
    shap_values   = shap_values_full,
    feature_names = feature_names_ohe,
)
print("Saved: outputs/results/phase5_shap_values_full.npz")

# Regenerate the importance CSV with the full-test-set values
shap_agg_full = aggregate_shap(shap_values_full)
mean_abs_full = np.abs(shap_agg_full).mean(axis=(0, 2))
importance_full = (pd.DataFrame({
        'feature': unique_originals,
        'mean_abs_shap_10k_sample': mean_abs_shap,
        'mean_abs_shap_full_test':  mean_abs_full,
    })
    .sort_values('mean_abs_shap_full_test', ascending=False)
    .reset_index(drop=True))
importance_full.to_csv(RESULTS_DIR / 'phase5_feature_importance.csv', index=False)

print("\nTop 10 features (full test set):")
print(importance_full.head(10).to_string(index=False))
print("\nPhase 5 complete.")

Computing SHAP values on full test set (92617 rows)...
  done in 25.1 min
  Full SHAP values shape: (92617, 398, 3)
Saved: outputs/results/phase5_shap_values_full.npz

Top 10 features (full test set):
         feature  mean_abs_shap_10k_sample  mean_abs_shap_full_test
   Accident_Type                  0.460920                 0.453142
        Latitude                  0.169328                 0.168629
     Cause_Group                  0.147606                 0.149074
  Police_Station                  0.104932                 0.105906
       Longitude                  0.104663                 0.103938
         Road_Km                  0.098599                 0.099172
            Hour                  0.083353                 0.082948
         Road_ID                  0.072190                 0.072243
Road_Delineation                  0.070487                 0.069936
       Day_Phase                  0.068080                 0.068096

Phase 5 complete.
